# SimMIM + VICReg SSL Pretraining — Swin Encoder (SwinUNETR)

**SimMIM + VICReg** hybrid SSL recipe on 3-channel synaptic microscopy
patches.

**Recipe**

For each training sample:

1. Two augmented views `view1`, `view2` are produced by the shared
   microscopy two-view pipeline (color/blur/elastic + random *D4* +
   per-channel z-score).
2. A single random block mask `m` is drawn.
3. Each view's masked pixels are replaced with a learnable per-channel
   `mask_token` (broadcast). The encoder + 1×1 Conv + PixelShuffle
   decoder reconstruct the *original* z-scored view at masked positions
   only.
4. The deepest encoder feature map is mean-pooled and projected through
   a 3-layer MLP (with BN+ReLU). VICReg's invariance / variance /
   covariance terms operate on the two pooled+projected embeddings.

**Loss**

$\mathcal{L} = w_\text{recon}
    \cdot \tfrac{1}{2} \big( \mathcal{L}_\text{simmim}(v_1)
                                + \mathcal{L}_\text{simmim}(v_2) \big)
   + w_\text{vicreg}
    \cdot \big( \lambda_\text{sim} \, \mathcal{L}_\text{sim}
                + \lambda_\text{std} \, \mathcal{L}_\text{std}
                + \lambda_\text{cov} \, \mathcal{L}_\text{cov} \big)$

where $\mathcal{L}_\text{simmim}$ is the L1 reconstruction error
restricted to masked pixels, and the VICReg terms are computed on the
projector outputs (fp32).

The decoder, projector, and mask token are all **discarded after
pretraining**. Only the encoder is transferred to SwinUNETR for
fine-tuning.

**References**
- Xie et al. *SimMIM: a Simple Framework for Masked Image Modeling*, CVPR 2022.
- Bardes, Ponce, LeCun *VICReg*, ICLR 2022.

The structure (sections 1–14), augmentation pipeline, weight loading,
LLRD utility, smoke run, training loop, and post-training inspection are
inherited unchanged from the master template.


## 1. Configuration

All knobs live in this section. The four config blocks below are intentionally
separate so that data / model / training / SSL hyperparameters can be edited
without scrolling.


In [ ]:
from types import SimpleNamespace


In [ ]:
cfg = SimpleNamespace(
    seed=42,
    output_root="./outputs",
    tag="template",
    # Where the encoder weights come from before SSL pretraining starts:
    #   "timm_imagenet" : load timm `swin_tiny_patch4_window7_224` (ImageNet)
    #   "simmim_local"  : load a local SimMIM checkpoint produced by
    #                     pretrain_simMIM_swin_v2_fixed.ipynb
    #   "scratch"       : random init (ablation baseline only)
    init_source="timm_imagenet",
    # Used purely for logging / save_dir naming. Does NOT branch any logic.
    method_name="METHOD",
    # If True, run the smoke training (1 epoch on a tiny subset) and exit
    # before the full loop. Useful for CI / iteration.
    dry_run=False,
)


In [ ]:
data_cfg = SimpleNamespace(
    data_root="../../data/patches_128",
    exclude_patterns=["KONTROLA"],
    val_split=0.1,
    batch_size=32,
    num_workers=2,
    pin_memory=True,
    # Canonical channel order. Asserted in the validity checks. Never permute.
    channel_names=["pre_synaptic", "post_synaptic", "structural"],
    # Computed in section 4 and persisted next to the checkpoints.
    channel_stats_path=None,
)


In [ ]:
# Architecture: matches timm `swin_tiny_patch4_window7_224` so its
# ImageNet weights load cleanly. Do NOT change feature_size / patch_size /
# depths / num_heads unless you are also switching the source checkpoint.
model_cfg = SimpleNamespace(
    in_channels=3,
    spatial_dims=2,
    img_size=128,
    feature_size=96,            # = embed_dim of timm Swin-T
    patch_size=4,               # = patch_size of timm Swin-T
    window_size=7,              # = window_size of timm Swin-T
    depths=(2, 2, 6, 2),        # = depths of timm Swin-T (NOTE: not 2,2,2,2)
    num_heads=(3, 6, 12, 24),   # = num_heads of timm Swin-T
    mlp_ratio=4.0,
    qkv_bias=True,
    dropout_path_rate=0.1,
    use_checkpoint=False,
    # timm model name when init_source == "timm_imagenet"
    timm_model_name="swin_tiny_patch4_window7_224",
    # Local checkpoint path when init_source == "simmim_local"
    simmim_ckpt_path=None,
)


In [ ]:
train_cfg = SimpleNamespace(
    epochs=200,
    warmup_epochs=10,
    # Encoder LR is multiplied by per-stage layer decay (see section 7.2).
    base_lr=1.5e-4,
    # Head LR is fixed (heads are randomly initialised, no LLRD needed).
    head_lr=1.5e-3,
    weight_decay=0.05,
    layer_decay=0.75,           # 0.75-0.9 typical for Swin/ViT fine-tuning
    grad_clip_norm=5.0,
    # Two-phase training: freeze encoder for the first N epochs so the
    # heads can warm up without blowing away pretrained features.
    freeze_encoder_epochs=2,
    # Naming
    experiment_name="simmim_vicreg_pretrain_swin",
    encoder_save_name="pretrained_encoder_simmim_vicreg.pt",
    # Smoke run (used by section 11)
    smoke_epochs=1,
    smoke_n=64,
    # Which key in validation_step's metrics dict drives "best" checkpoint.
    val_metric_key="ssl_loss",
    # Direction: "min" or "max" — controls best-checkpoint selection.
    val_metric_direction="min",
)


In [ ]:
# SimMIM + VICReg hyperparameters. Read by build_heads / compute_loss /
# validation_step below.
ssl_cfg = SimpleNamespace(
    # --- SimMIM masking ---
    mask_ratio=0.4,                  # fraction of mask blocks set to 1
    mask_block_size=16,              # pixel block size; img_size / block_size
                                     # gives the mask grid dim (must divide).
                                     # img_size=128, block=16 -> 8x8 mask grid.

    # --- SimMIM reconstruction loss ---
    # 'l1'        : L1 on masked pixels only (recommended for z-scored target).
    # 'l1_l2_mix' : 0.5*L1 + 0.5*L2 on masked pixels.
    # NOTE: the 'l1_fg' (foreground-weighted) variant from training_v4 is not
    # exposed here because the foreground threshold tau is defined in raw
    # [0, 1] image space; the v2 template feeds z-scored views.
    loss_kind="l1",

    # --- VICReg ---
    lambda_sim=25.0,                 # invariance term (MSE between projections)
    lambda_std=25.0,                 # variance hinge term
    lambda_cov=1.0,                  # covariance off-diagonal term

    # --- Loss combination ---
    w_recon=1.0,                     # weight of the SimMIM reconstruction term
    w_vicreg=1.0,                    # weight of the VICReg total

    # --- Projector head (VICReg style: MLP -> BN -> ReLU x2 -> Linear) ---
    projector_hidden=1024,           # hidden width (also output of the 1st BN block)
    projector_dim=1024,              # output embedding dim used by VICReg

    # --- (unused for SimMIM+VICReg) ---
    use_ema=False,                   # this recipe does not use a target/teacher encoder
)


## 2. Imports & Device


In [ ]:
import os
import sys
import csv
import json
import math
import time
import copy
import itertools
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from tqdm.auto import tqdm

from monai.networks.nets.swin_unetr import SwinTransformer

sys.path.insert(0, os.path.abspath("../.."))
from data_utils.patch_dataset import PatchDataset


In [ ]:
# Prevent OpenBLAS from spawning too many threads (causes hangs on HPC)
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK",
             os.environ.get("PBS_NUM_PPN", 4)))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(max(1, n_cpus // 2)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator().manual_seed(cfg.seed)
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")


## 3. Data Loading


In [ ]:
dataset = PatchDataset(
    root=data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
print(f"Total patches: {len(dataset)}")

sample = dataset[0]
print(f"Sample shape: {sample.shape}, dtype: {sample.dtype}")
print(f"Value range:  [{sample.min():.4f}, {sample.max():.4f}]")
assert sample.ndim == 3, f"Expected 3D tensor (C,H,W), got {sample.ndim}D"
assert sample.shape[0] == model_cfg.in_channels, (
    f"Dataset has {sample.shape[0]} channels, model_cfg expects {model_cfg.in_channels}"
)
assert sample.shape[-1] == model_cfg.img_size, (
    f"Dataset patch size {sample.shape[-1]} does not match model_cfg.img_size {model_cfg.img_size}"
)
assert 0 <= sample.min() and sample.max() <= 1.0 + 1e-6, "Values outside [0, 1]"


In [ ]:
n_val = int(len(dataset) * data_cfg.val_split)
n_train = len(dataset) - n_val
train_subset, val_subset = random_split(
    dataset, [n_train, n_val], generator=generator
)
data_cfg.n_train = n_train
data_cfg.n_val = n_val
print(f"Train: {n_train}, Validation: {n_val}")


## 4. Per-Channel Statistics

Computed on the training split **before any augmentation**. Persisted next
to the checkpoints so downstream fine-tuning notebooks reuse identical stats.
Never reuse ImageNet statistics for fluorescence microscopy.


In [ ]:
def _compute_channel_stats(subset, max_samples=4096):
    """Streaming mean/std per channel over a Subset of (C,H,W) tensors."""
    n = min(len(subset), max_samples)
    sums = torch.zeros(model_cfg.in_channels, dtype=torch.float64)
    sqsums = torch.zeros(model_cfg.in_channels, dtype=torch.float64)
    count = 0
    for i in tqdm(range(n), desc="channel stats"):
        x = subset[i].double()  # (C,H,W)
        sums += x.sum(dim=(1, 2))
        sqsums += (x ** 2).sum(dim=(1, 2))
        count += x.shape[1] * x.shape[2]
    mean = (sums / count).float()
    var = (sqsums / count).float() - mean ** 2
    std = var.clamp_min(1e-12).sqrt()
    return mean, std


CH_MEAN, CH_STD = _compute_channel_stats(train_subset)
for name, m, s in zip(data_cfg.channel_names, CH_MEAN.tolist(), CH_STD.tolist()):
    print(f"  {name:15s}  mean={m:.5f}  std={s:.5f}")


## 5. Two-View Augmentation Pipeline

 dihedral group (flips + 90° rotations) + small affine +
mild elastic + **global** photometric jitter (same factor across channels)
+ small per-channel additive offset + Poisson-Gaussian noise + per-channel
z-score normalization.

Geometric transforms are applied identically across all 3 channels within a
view; the two views get independent random parameters. No per-channel
multiplicative jitter, no chromatic shift, no channel dropout — those would
destroy the synaptic co-localization signal.


In [ ]:
class MicroscopyTwoViewTransform:
    """Two-view augmentation for SSL on 3-channel synaptic microscopy.

    Input  : (C, H, W) float32 tensor in [0, 1].
    Output : (view1, view2) — each (C, H, W) float32, post-normalization.
    """

    def __init__(
        self,
        ch_mean: torch.Tensor,
        ch_std: torch.Tensor,
        *,
        elastic_alpha: float = 30.0,
        elastic_sigma: float = 6.0,
        elastic_p: float = 0.3,
        translate_max: int = 5,
        global_brightness: float = 0.20,
        global_contrast: float = 0.15,
        global_gamma_range: tuple = (0.85, 1.20),
        per_channel_offset_max: float = 0.03,
        gauss_noise_sigma: float = 0.015,
        poisson_scale: float = 0.02,
        erase_p: float = 0.25,
        erase_max_frac: float = 0.08,
    ):
        self.ch_mean = ch_mean.view(-1, 1, 1)
        self.ch_std = ch_std.view(-1, 1, 1)
        self.elastic_alpha = elastic_alpha
        self.elastic_sigma = elastic_sigma
        self.elastic_p = elastic_p
        self.translate_max = translate_max
        self.global_brightness = global_brightness
        self.global_contrast = global_contrast
        self.global_gamma_range = global_gamma_range
        self.per_channel_offset_max = per_channel_offset_max
        self.gauss_noise_sigma = gauss_noise_sigma
        self.poisson_scale = poisson_scale
        self.erase_p = erase_p
        self.erase_max_frac = erase_max_frac

    # ---------- geometric (applied identically across channels) ----------
    @staticmethod
    def _flips_rot90(x):
        if torch.rand(()) < 0.5:
            x = torch.flip(x, dims=(-1,))
        if torch.rand(()) < 0.5:
            x = torch.flip(x, dims=(-2,))
        k = int(torch.randint(0, 4, ()).item())
        if k:
            x = torch.rot90(x, k=k, dims=(-2, -1))
        return x

    def _affine_translate(self, x):
        """Small integer translation with reflective padding."""
        if self.translate_max <= 0:
            return x
        tx = int(torch.randint(-self.translate_max, self.translate_max + 1, ()).item())
        ty = int(torch.randint(-self.translate_max, self.translate_max + 1, ()).item())
        if tx == 0 and ty == 0:
            return x
        pad = self.translate_max
        x = F.pad(x.unsqueeze(0), (pad, pad, pad, pad), mode="reflect").squeeze(0)
        H, W = x.shape[-2:]
        return x[..., pad - ty:H - pad - ty, pad - tx:W - pad - tx]

    def _elastic(self, x):
        if torch.rand(()) >= self.elastic_p:
            return x
        C, H, W = x.shape
        # Random displacement field, smoothed by a Gaussian via separable conv.
        dx = (torch.rand(1, 1, H, W) * 2 - 1)
        dy = (torch.rand(1, 1, H, W) * 2 - 1)
        k = max(3, int(self.elastic_sigma * 3) | 1)  # odd
        kernel_1d = torch.exp(-((torch.arange(k) - k // 2) ** 2) / (2 * self.elastic_sigma ** 2))
        kernel_1d = kernel_1d / kernel_1d.sum()
        kx = kernel_1d.view(1, 1, 1, k)
        ky = kernel_1d.view(1, 1, k, 1)
        dx = F.conv2d(F.conv2d(dx, kx, padding=(0, k // 2)), ky, padding=(k // 2, 0))
        dy = F.conv2d(F.conv2d(dy, kx, padding=(0, k // 2)), ky, padding=(k // 2, 0))
        dx = dx.squeeze() * self.elastic_alpha
        dy = dy.squeeze() * self.elastic_alpha
        # Build a sampling grid.
        gy, gx = torch.meshgrid(
            torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing="ij"
        )
        grid = torch.stack(
            (gx + 2 * dx / W, gy + 2 * dy / H), dim=-1
        ).unsqueeze(0)
        return F.grid_sample(
            x.unsqueeze(0), grid, mode="bilinear",
            padding_mode="reflection", align_corners=True,
        ).squeeze(0)

    def _random_erase(self, x):
        if torch.rand(()) >= self.erase_p:
            return x
        C, H, W = x.shape
        area = H * W
        target_area = float(torch.empty(1).uniform_(0.01, self.erase_max_frac).item()) * area
        aspect = float(torch.empty(1).uniform_(0.5, 2.0).item())
        h = int(round(math.sqrt(target_area * aspect)))
        w = int(round(math.sqrt(target_area / aspect)))
        if h < 1 or w < 1 or h >= H or w >= W:
            return x
        top = int(torch.randint(0, H - h, ()).item())
        left = int(torch.randint(0, W - w, ()).item())
        x = x.clone()
        x[:, top:top + h, left:left + w] = 0
        return x

    # ---------- photometric (GLOBAL: same factor across channels) ----------
    def _global_brightness_contrast(self, x):
        b = 1.0 + (torch.rand(()) * 2 - 1) * self.global_brightness
        c = 1.0 + (torch.rand(()) * 2 - 1) * self.global_contrast
        mean = x.mean()
        return ((x - mean) * c + mean) * b

    def _global_gamma(self, x):
        lo, hi = self.global_gamma_range
        g = torch.empty(()).uniform_(lo, hi).item()
        return x.clamp_min(0).pow(g)

    # ---------- photometric (per-channel additive ONLY) ----------
    def _per_channel_offset(self, x):
        offs = (torch.rand(x.shape[0], 1, 1) * 2 - 1) * self.per_channel_offset_max
        return x + offs

    # ---------- noise ----------
    def _poisson_gaussian_noise(self, x):
        # Poisson shot noise (per-pixel, per-channel — physically accurate)
        if self.poisson_scale > 0:
            scale = 1.0 / max(self.poisson_scale, 1e-8)
            x = torch.poisson((x.clamp_min(0) * scale).double()).float() / scale
        # Read noise (Gaussian)
        if self.gauss_noise_sigma > 0:
            x = x + torch.randn_like(x) * self.gauss_noise_sigma
        return x

    # ---------- pipeline ----------
    def _one_view(self, x):
        x = self._flips_rot90(x)
        x = self._affine_translate(x)
        x = self._elastic(x)
        x = self._random_erase(x)
        x = self._global_brightness_contrast(x)
        x = self._global_gamma(x)
        x = self._per_channel_offset(x)
        x = self._poisson_gaussian_noise(x)
        x = (x - self.ch_mean) / self.ch_std
        return x

    def __call__(self, x):
        if not torch.is_tensor(x):
            x = torch.from_numpy(x)
        x = x.float()
        return self._one_view(x), self._one_view(x)


class _ValSingleViewTransform:
    """Validation transform: just per-channel z-score, no augmentation."""

    def __init__(self, ch_mean, ch_std):
        self.ch_mean = ch_mean.view(-1, 1, 1)
        self.ch_std = ch_std.view(-1, 1, 1)

    def __call__(self, x):
        if not torch.is_tensor(x):
            x = torch.from_numpy(x)
        return ((x.float() - self.ch_mean) / self.ch_std)


In [ ]:
class _AugmentedSubset(torch.utils.data.Dataset):
    """Wrap a torch.utils.data.Subset and apply a transform on the fly."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        return self.transform(self.subset[i])


train_transform = MicroscopyTwoViewTransform(CH_MEAN, CH_STD)
val_transform = _ValSingleViewTransform(CH_MEAN, CH_STD)

train_ds = _AugmentedSubset(train_subset, train_transform)
val_ds = _AugmentedSubset(val_subset, val_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=data_cfg.batch_size,
    shuffle=True,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_cfg.batch_size,
    shuffle=False,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 6. Encoder Construction & Pretrained Weight Loading

The encoder is `monai.networks.nets.swin_unetr.SwinTransformer` configured
to match `swin_tiny_patch4_window7_224` so timm's ImageNet weights load
cleanly via `convert_timm_to_swinunetr_state_dict`.

The remapping is heuristic-based: it walks the timm state-dict and rewrites
keys to MONAI's naming (`layers.{i}` → `layers{i+1}`, etc.). Every loaded /
skipped / randomly-initialised parameter is logged so you can spot a silent
mismatch.


In [ ]:
def build_swin_encoder(cfg_m: SimpleNamespace) -> nn.Module:
    """Construct the MONAI SwinTransformer encoder from `model_cfg`."""
    patch_size = (cfg_m.patch_size,) * cfg_m.spatial_dims
    window_size = (cfg_m.window_size,) * cfg_m.spatial_dims
    return SwinTransformer(
        in_chans=cfg_m.in_channels,
        embed_dim=cfg_m.feature_size,
        window_size=window_size,
        patch_size=patch_size,
        depths=list(cfg_m.depths),
        num_heads=list(cfg_m.num_heads),
        mlp_ratio=cfg_m.mlp_ratio,
        qkv_bias=cfg_m.qkv_bias,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=cfg_m.dropout_path_rate,
        norm_layer=nn.LayerNorm,
        use_checkpoint=cfg_m.use_checkpoint,
        spatial_dims=cfg_m.spatial_dims,
    )


In [ ]:
def _remap_timm_key_to_monai(k: str) -> str | None:
    """Rewrite a timm Swin key to a MONAI SwinTransformer key.

    Returns None for keys that have no MONAI equivalent (classifier head,
    final norm, timm's stage-0 Identity downsample, etc.).

    Naming differences between timm and MONAI's SwinTransformer:
      * Stage container:
          timm  uses  `layers.{i}.*`           (nn.Sequential of stages)
          MONAI uses  `layers{i+1}.0.*`        (named attrs holding nn.ModuleList)
      * Downsample placement INSIDE a stage:
          timm  : downsample is at the START of stage i and operates on the
                  stage's INPUT channels — `layers.{i}.downsample` projects
                  C_{i-1} → C_i  (and `layers.0.downsample` is `Identity`,
                  so it does not appear in the state-dict).
          MONAI : downsample is at the END   of stage i and operates on the
                  stage's OUTPUT channels — `layersI.0.downsample` projects
                  C_i → C_{i+1}.
        Net effect: timm `layers.{i}.downsample` and MONAI `layers{i}.0.downsample`
        carry IDENTICAL weights (both project C_{i-1} → C_i), but their stage
        index differs by ONE relative to the blocks they accompany.
            timm layers.{i}.blocks.*       → MONAI layers{i+1}.0.blocks.*
            timm layers.{i}.downsample.*   → MONAI layers{i  }.0.downsample.*
        (i ∈ {1,2,3} for downsample; i ∈ {0,1,2,3} for blocks.)
        MONAI's `layers4.0.downsample` (768→1536 in Swin-T) has NO timm
        counterpart and stays randomly-initialized — that is expected.
      * MLP attribute names inside each block:
          timm  : `mlp.fc1`, `mlp.fc2`
          MONAI : `mlp.linear1`, `mlp.linear2`
      * `relative_position_index`:
          timm  : non-persistent buffer  → NOT in timm state_dict
          MONAI : persistent buffer      → IS in MONAI state_dict, but it's a
                  deterministic function of `window_size`, so MONAI's default
                  value is already correct and we don't need to copy anything.
    """
    # 1) Drop classifier / final-norm / pooling / aux-head keys.
    if k.startswith(("head.", "norm.", "norm_pre.", "norm_post.", "pre_logits.")):
        return None

    # 2) Patch embed: identical naming in both, no rewrite needed.
    if k.startswith("patch_embed."):
        return k

    # 3) Stage-level keys.
    if k.startswith("layers."):
        rest = k[len("layers."):]
        idx_str, sep, tail = rest.partition(".")
        if not sep:
            return None
        try:
            i = int(idx_str)
        except ValueError:
            return None

        head, _, _ = tail.partition(".")
        if head == "blocks":
            new_key = f"layers{i + 1}.0.{tail}"
        elif head == "downsample":
            # timm stage 0 has Identity downsample; nothing to copy.
            if i == 0:
                return None
            new_key = f"layers{i}.0.{tail}"
        else:
            # Anything else under a stage (e.g., a per-stage norm) — unknown.
            return None

        # MLP rename: only inside SwinTransformerBlock, i.e. under blocks.
        if ".blocks." in new_key:
            new_key = (
                new_key
                .replace(".mlp.fc1.", ".mlp.linear1.")
                .replace(".mlp.fc2.", ".mlp.linear2.")
            )
        return new_key

    # 4) Anything else is unexpected — drop it conservatively.
    return None


def _adapt_first_conv(weight: torch.Tensor, target_in_chans: int) -> torch.Tensor:
    """Adapt patch_embed.proj.weight from N_src input channels to N_tgt.

    Strategy: average across source input channels and tile to the target
    count, then rescale by N_src/N_tgt so the per-channel sum is preserved.
    This is the standard initialization used by timm/MONAI/MMSeg when the
    input channel count differs from the pretraining one.
    """
    src_in = weight.shape[1]
    if src_in == target_in_chans:
        return weight
    avg = weight.mean(dim=1, keepdim=True)                  # (out, 1, kh, kw)
    tiled = avg.repeat(1, target_in_chans, 1, 1)            # (out, N_tgt, kh, kw)
    return tiled * (src_in / target_in_chans)


def convert_timm_to_swinunetr_state_dict(
    timm_sd: dict[str, torch.Tensor],
    target_sd: dict[str, torch.Tensor],
    target_in_chans: int,
) -> tuple[dict[str, torch.Tensor], dict]:
    """Remap a timm Swin state-dict to MONAI's SwinTransformer naming.

    Returns
    -------
    new_sd : dict
        Re-keyed state-dict ready for `model.load_state_dict(new_sd, strict=False)`.
    summary : dict
        loaded / skipped / shape_mismatch / random_init counters with detail lists.
    """
    new_sd: dict[str, torch.Tensor] = {}
    skipped, shape_mismatch, copied = [], [], []

    for k, v in timm_sd.items():
        nk = _remap_timm_key_to_monai(k)
        if nk is None:
            skipped.append((k, "no_monai_equivalent"))
            continue
        if nk not in target_sd:
            skipped.append((k, f"missing_in_target:{nk}"))
            continue
        # Adapt first conv if input channel count differs.
        if nk.endswith("patch_embed.proj.weight") and v.dim() == 4:
            v = _adapt_first_conv(v, target_in_chans)
        if v.shape != target_sd[nk].shape:
            shape_mismatch.append((k, nk, tuple(v.shape), tuple(target_sd[nk].shape)))
            continue
        new_sd[nk] = v
        copied.append(nk)

    in_target_not_loaded = sorted(set(target_sd) - set(new_sd))
    summary = {
        "n_target_params":      len(target_sd),
        "n_loaded":             len(copied),
        "n_skipped_in_source":  len(skipped),
        "n_shape_mismatch":     len(shape_mismatch),
        "n_random_init":        len(in_target_not_loaded),
        "skipped":              skipped,
        "shape_mismatch":       shape_mismatch,
        "random_init":          in_target_not_loaded,
    }
    return new_sd, summary


def _print_load_summary(summary: dict, verbose: bool = False) -> None:
    print(f"  loaded                 : {summary['n_loaded']} / {summary['n_target_params']}")
    print(f"  random-init in target  : {summary['n_random_init']}")
    print(f"  skipped from source    : {summary['n_skipped_in_source']}")
    print(f"  shape mismatches       : {summary['n_shape_mismatch']}")
    if verbose:
        if summary["shape_mismatch"]:
            print("  --- shape mismatches ---")
            for entry in summary["shape_mismatch"][:20]:
                k, nk, src, tgt = entry
                print(f"    {k} -> {nk}  src={src}  tgt={tgt}")
        if summary["random_init"]:
            print("  --- random-init keys (first 20) ---")
            for k in summary["random_init"][:20]:
                print(f"    {k}")
        if summary["skipped"]:
            print("  --- skipped from source (first 20) ---")
            for entry in summary["skipped"][:20]:
                k, reason = entry
                print(f"    {k}  ({reason})")


In [ ]:
def load_pretrained_into_encoder(
    encoder: nn.Module,
    init_source: str,
    model_cfg_: SimpleNamespace,
) -> dict:
    """Dispatch on `init_source` and load weights into `encoder` in place."""
    if init_source == "scratch":
        print("[init] scratch — keeping random initialization")
        return {"n_loaded": 0, "n_random_init": len(encoder.state_dict()), "source": "scratch"}

    target_sd = encoder.state_dict()

    if init_source == "timm_imagenet":
        try:
            import timm
        except ImportError as e:
            raise ImportError(
                "timm is required for init_source='timm_imagenet'. "
                "Install with: pip install timm"
            ) from e
        print(f"[init] downloading timm `{model_cfg_.timm_model_name}` ...")
        timm_model = timm.create_model(
            model_cfg_.timm_model_name, pretrained=True, num_classes=0
        )
        timm_sd = timm_model.state_dict()
        new_sd, summary = convert_timm_to_swinunetr_state_dict(
            timm_sd, target_sd, target_in_chans=model_cfg_.in_channels
        )
    elif init_source == "simmim_local":
        if not model_cfg_.simmim_ckpt_path:
            raise ValueError(
                "init_source='simmim_local' but model_cfg.simmim_ckpt_path is None"
            )
        print(f"[init] loading SimMIM checkpoint: {model_cfg_.simmim_ckpt_path}")
        ckpt = torch.load(model_cfg_.simmim_ckpt_path, map_location="cpu")
        # Accept either a raw state_dict or a checkpoint dict from
        # pretrain_simMIM_swin_v2_fixed.ipynb.
        if "model_state_dict" in ckpt:
            full_sd = ckpt["model_state_dict"]
            # Strip "swinViT." prefix from SimMIM wrapper keys if present.
            simmim_sd = {
                k[len("swinViT."):]: v
                for k, v in full_sd.items()
                if k.startswith("swinViT.")
            }
            if not simmim_sd:
                simmim_sd = full_sd
        else:
            simmim_sd = ckpt
        # MONAI → MONAI, so naming is already aligned. Filter out any
        # decoder / mask_token keys that don't belong to the encoder.
        new_sd = {k: v for k, v in simmim_sd.items() if k in target_sd}
        copied = list(new_sd.keys())
        in_target_not_loaded = sorted(set(target_sd) - set(new_sd))
        summary = {
            "n_target_params":      len(target_sd),
            "n_loaded":             len(copied),
            "n_skipped_in_source":  len(set(simmim_sd) - set(target_sd)),
            "n_shape_mismatch":     0,
            "n_random_init":        len(in_target_not_loaded),
            "skipped":              sorted(set(simmim_sd) - set(target_sd)),
            "shape_mismatch":       [],
            "random_init":          in_target_not_loaded,
        }
    else:
        raise ValueError(
            f"Unknown init_source={init_source!r}. "
            "Use 'timm_imagenet' / 'simmim_local' / 'scratch'."
        )

    incompatible = encoder.load_state_dict(new_sd, strict=False)
    summary["unexpected_keys"] = list(incompatible.unexpected_keys)
    summary["missing_keys"]    = list(incompatible.missing_keys)
    summary["source"]          = init_source

    print(f"[init] source={init_source}")
    _print_load_summary(summary, verbose=False)

    # Loud failure if essentially nothing loaded but the user asked for pretrained weights.
    if init_source != "scratch" and summary["n_loaded"] < 0.5 * summary["n_target_params"]:
        warnings.warn(
            f"Only {summary['n_loaded']}/{summary['n_target_params']} encoder "
            f"params were loaded from {init_source}. The remap may have failed; "
            "inspect summary['random_init'] and summary['shape_mismatch']."
        )
    return summary


In [ ]:
encoder = build_swin_encoder(model_cfg).to(device)
load_summary = load_pretrained_into_encoder(encoder, cfg.init_source, model_cfg)

n_total = sum(p.numel() for p in encoder.parameters())
print(f"Encoder params: {n_total / 1e6:.2f} M")


## 7. Method definitions

### 7.1 `build_heads`

In [ ]:
# === SimMIM + VICReg: 1x1 Conv + PixelShuffle decoder, learnable
# === per-channel mask token, and 3-layer MLP projector.

class _SimMIMDecoder(nn.Module):
    """1x1 Conv + PixelShuffle: (B, enc_ch, S, S) -> (B, in_channels, H, W).

    Canonical SimMIM-style decoder. The output is *not* sigmoid'd because
    the reconstruction target is the post-normalization z-scored view,
    which is unbounded.
    """

    def __init__(self, enc_ch: int, out_channels: int, encoder_stride: int):
        super().__init__()
        self.encoder_stride = encoder_stride
        self.up = nn.Sequential(
            nn.Conv2d(enc_ch, encoder_stride * encoder_stride * out_channels,
                      kernel_size=1),
            nn.PixelShuffle(encoder_stride),
        )

    def forward(self, z):
        return self.up(z)


class _MaskToken(nn.Module):
    """Wraps a learnable per-channel mask token of shape (1, C, 1, 1).

    Wrapped in an nn.Module so the template's heads-as-dict bookkeeping
    (state_dict / .parameters() / .to(device)) treats it uniformly with
    the decoder and projector.
    """

    def __init__(self, in_channels: int):
        super().__init__()
        self.token = nn.Parameter(torch.zeros(1, in_channels, 1, 1))
        nn.init.trunc_normal_(self.token, mean=0.0, std=0.02)

    def forward(self):
        return self.token


class _Projector(nn.Module):
    """VICReg-style projector: Linear -> BN -> ReLU -> Linear -> BN -> ReLU -> Linear.

    The bias on the inner Linear layers is disabled (BN absorbs it). The
    final Linear retains a bias.
    """

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim, bias=True),
        )

    def forward(self, z_pool):
        return self.net(z_pool)


def build_heads(encoder: nn.Module, cfg_ssl: SimpleNamespace):
    """SimMIM + VICReg heads: decoder + mask token + projector (as a dict).

    Returned as a dict so the template's training loop iterates parameters
    via heads.values() and saves/loads each module's state_dict
    independently.
    """
    # MONAI SwinTransformer.forward returns [x0_out, ..., x4_out] (5 stages):
    # an extra patch-merging is applied after the last BasicLayer, so the
    # deepest feature map (selected by `_encode_deepest`) has
    # `feature_size * 2 ** len(depths)` channels at spatial
    # `img_size / (patch_size * 2 ** len(depths))`.
    enc_ch = model_cfg.feature_size * (2 ** len(model_cfg.depths))
    enc_spatial = model_cfg.img_size // (model_cfg.patch_size * (2 ** len(model_cfg.depths)))
    encoder_stride = model_cfg.patch_size * (2 ** len(model_cfg.depths))
    if encoder_stride * enc_spatial != model_cfg.img_size:
        raise ValueError(
            f"img_size={model_cfg.img_size} not divisible by "
            f"encoder_stride={encoder_stride} (enc_spatial={enc_spatial})."
        )
    if model_cfg.img_size % cfg_ssl.mask_block_size != 0:
        raise ValueError(
            f"img_size={model_cfg.img_size} not divisible by "
            f"mask_block_size={cfg_ssl.mask_block_size}."
        )

    decoder = _SimMIMDecoder(
        enc_ch=enc_ch,
        out_channels=model_cfg.in_channels,
        encoder_stride=encoder_stride,
    )
    mask_token = _MaskToken(model_cfg.in_channels)
    projector = _Projector(
        in_dim=enc_ch,
        hidden_dim=cfg_ssl.projector_hidden,
        out_dim=cfg_ssl.projector_dim,
    )
    return {"decoder": decoder, "mask_token": mask_token, "projector": projector}


### 7.2 Layer-wise LR decay utility

In [ ]:
def param_groups_layer_decay(
    encoder: nn.Module,
    base_lr: float,
    weight_decay: float,
    layer_decay: float,
    no_decay_keywords: tuple = (
        "bias", "norm", "relative_position_bias_table", "absolute_pos_embed",
        "mask_token",
    ),
) -> list[dict]:
    """Build per-parameter-group LR / weight-decay for a MONAI SwinTransformer.

    Stage assignment used by the decay multiplier:
        depth 0 : patch_embed
        depth d : layers{d}            (1..4)
        depth 5 : final norm           (no LLRD scaling — full LR)
    Decay multiplier for stage d is `layer_decay ** (5 - d)` so the deepest
    stage learns at full LR and the patch embed at the smallest LR.
    """
    n_stages = 5  # patch_embed + 4 stages
    scales = [layer_decay ** (n_stages - d) for d in range(n_stages + 1)]

    def assign_stage(name: str) -> int:
        if name.startswith("patch_embed"):
            return 0
        for i in range(1, 5):
            if name.startswith(f"layers{i}"):
                return i
        return 5  # everything else (final norm, etc.)

    groups: dict[tuple, dict] = {}
    for name, p in encoder.named_parameters():
        if not p.requires_grad:
            continue
        stage = assign_stage(name)
        no_decay = any(kw in name for kw in no_decay_keywords)
        key = (stage, no_decay)
        if key not in groups:
            groups[key] = {
                "params": [],
                "lr": base_lr * scales[stage],
                "weight_decay": 0.0 if no_decay else weight_decay,
                "stage": stage,
                "no_decay": no_decay,
            }
        groups[key]["params"].append(p)
    return list(groups.values())


### 7.3 `build_optimizer_and_scheduler`

In [ ]:
def make_lr_lambda(warmup_epochs: int, total_epochs: int):
    """Linear warmup for `warmup_epochs` then cosine decay to 0."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_lambda


# === SimMIM + VICReg: AdamW with LLRD on encoder + flat LR on heads ===
def build_optimizer_and_scheduler(encoder, heads, cfg_t):
    encoder_groups = param_groups_layer_decay(
        encoder,
        base_lr=cfg_t.base_lr,
        weight_decay=cfg_t.weight_decay,
        layer_decay=cfg_t.layer_decay,
    )
    head_params = (
        list(heads.parameters()) if isinstance(heads, nn.Module)
        else [p for h in heads.values() for p in h.parameters()]
    )
    head_group = {"params": head_params, "lr": cfg_t.head_lr,
                  "weight_decay": cfg_t.weight_decay}
    optimizer = torch.optim.AdamW(
        encoder_groups + [head_group], betas=(0.9, 0.999),
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, make_lr_lambda(cfg_t.warmup_epochs, cfg_t.epochs),
    )
    return optimizer, scheduler


### 7.4 `compute_loss`

In [ ]:
# === SimMIM + VICReg: masked reconstruction + variance/invariance/covariance ===

def _encode_deepest(encoder: nn.Module, x: torch.Tensor) -> torch.Tensor:
    """Wrap encoder forward to return the deepest feature map (B, C, S, S)."""
    return encoder(x.contiguous())[-1]


def _random_block_mask(img: torch.Tensor, block_size: int, mask_ratio: float) -> torch.Tensor:
    """Per-sample grid mask on a (B, C, H, W) image; returns (B, 1, H, W) of {0, 1}.

    Each sample independently gets a random subset of grid cells masked.
    The mask is broadcast across channels.
    """
    B, _, H, W = img.shape
    if H % block_size or W % block_size:
        raise ValueError(f"block_size={block_size} must divide H={H} and W={W}.")
    gh, gw = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask = max(1, int(round(n_blocks * mask_ratio)))
    noise = torch.rand(B, n_blocks, device=img.device)
    rank = noise.argsort(dim=1)
    # rank < n_mask => block is masked.
    flat = (rank < n_mask).to(img.dtype)
    grid = flat.view(B, 1, gh, gw)
    return F.interpolate(grid, scale_factor=block_size, mode="nearest")


def _apply_mask(view: torch.Tensor, mask: torch.Tensor, mask_token_module) -> torch.Tensor:
    """Replace pixels in `view` (B, C, H, W) where `mask` (B, 1, H, W) is 1
    with the broadcast per-channel mask_token from `mask_token_module()`.
    """
    return view * (1.0 - mask) + mask_token_module() * mask


def _simmim_recon_l1(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """L1 reconstruction loss on the masked region only.

    Note: for a z-scored target, the absolute scale is in std-deviation
    units rather than in [0, 1] as in raw-pixel SimMIM, but the gradient
    direction is identical.
    """
    err = (pred - target).abs() * mask
    denom = mask.sum() * pred.shape[1] + 1e-8
    return err.sum() / denom


def _simmim_recon_l1_l2(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """0.5 * L1 + 0.5 * L2 on the masked region only."""
    err1 = (pred - target).abs() * mask
    err2 = (pred - target).pow(2) * mask
    denom = mask.sum() * pred.shape[1] + 1e-8
    return 0.5 * (err1.sum() / denom) + 0.5 * (err2.sum() / denom)


def reconstruction_loss(pred: torch.Tensor, target: torch.Tensor,
                        mask: torch.Tensor, kind: str) -> torch.Tensor:
    if kind == "l1":
        return _simmim_recon_l1(pred, target, mask)
    if kind == "l1_l2_mix":
        return _simmim_recon_l1_l2(pred, target, mask)
    raise ValueError(f"Unknown loss_kind={kind!r}; expected 'l1' or 'l1_l2_mix'.")


def _vicreg_terms(z1: torch.Tensor, z2: torch.Tensor, eps: float = 1e-4):
    """Compute VICReg's (sim, std, cov) terms in fp32.

    sim = MSE(z1, z2)
    std = relu(1 - sqrt(var + eps)).mean(), averaged over the two views.
    cov = sum_off_diag(cov_matrix^2) / D, averaged over the two views.
    """
    z1 = z1.float()
    z2 = z2.float()
    B, D = z1.shape
    L_sim = F.mse_loss(z1, z2)

    std1 = torch.sqrt(z1.var(dim=0, unbiased=False) + eps)
    std2 = torch.sqrt(z2.var(dim=0, unbiased=False) + eps)
    L_std = 0.5 * (F.relu(1.0 - std1).mean() + F.relu(1.0 - std2).mean())

    z1c = z1 - z1.mean(dim=0, keepdim=True)
    z2c = z2 - z2.mean(dim=0, keepdim=True)
    denom = max(B - 1, 1)
    cov1 = (z1c.T @ z1c) / denom
    cov2 = (z2c.T @ z2c) / denom
    off1 = cov1.pow(2).sum() - cov1.diagonal().pow(2).sum()
    off2 = cov2.pow(2).sum() - cov2.diagonal().pow(2).sum()
    L_cov = (off1 + off2) / (2.0 * D)
    return L_sim, L_std, L_cov


def compute_loss(encoder, heads, view1, view2, cfg_ssl):
    decoder    = heads["decoder"]
    mask_token = heads["mask_token"]
    projector  = heads["projector"]

    # One mask shared between the two views (matches v4 simmim_vicreg).
    mask = _random_block_mask(view1, cfg_ssl.mask_block_size, cfg_ssl.mask_ratio)

    # Replace masked pixels with the mask token, then encode + decode.
    v1_masked = _apply_mask(view1, mask, mask_token)
    v2_masked = _apply_mask(view2, mask, mask_token)
    z1 = _encode_deepest(encoder, v1_masked)
    z2 = _encode_deepest(encoder, v2_masked)
    r1 = decoder(z1)
    r2 = decoder(z2)

    # SimMIM reconstruction (averaged over the two views), masked only.
    L_recon = 0.5 * (
        reconstruction_loss(r1, view1, mask, cfg_ssl.loss_kind)
        + reconstruction_loss(r2, view2, mask, cfg_ssl.loss_kind)
    )

    # VICReg on the projector outputs (fp32 for projector + loss).
    z1_pool = z1.mean(dim=(-2, -1))
    z2_pool = z2.mean(dim=(-2, -1))
    p1 = projector(z1_pool.float())
    p2 = projector(z2_pool.float())
    L_sim, L_std, L_cov = _vicreg_terms(p1, p2)
    L_vicreg = (cfg_ssl.lambda_sim * L_sim
                + cfg_ssl.lambda_std * L_std
                + cfg_ssl.lambda_cov * L_cov)

    rec_w = float(getattr(cfg_ssl, "w_recon", 1.0))
    vic_w = float(getattr(cfg_ssl, "w_vicreg", 1.0))
    loss = rec_w * L_recon + vic_w * L_vicreg

    metrics = {
        "ssl_loss": float(loss.detach().item()),
        "recon":    float(L_recon.detach().item()),
        "sim":      float(L_sim.detach().item()),
        "std":      float(L_std.detach().item()),
        "cov":      float(L_cov.detach().item()),
        "vicreg":   float(L_vicreg.detach().item()),
    }
    return loss, metrics


### 7.5 `validation_step`

In [ ]:
# === SimMIM + VICReg: per-batch validation (single view, no augmentation) ===
# The val_loader yields a single z-scored tensor per batch (no two-view
# augmentation), so VICReg cannot be evaluated. We report only the SimMIM
# reconstruction term, which is also assigned to `ssl_loss` so it is
# directly comparable across runs and used for best-checkpoint selection.
def validation_step(encoder, heads, batch, cfg_ssl):
    decoder    = heads["decoder"]
    mask_token = heads["mask_token"]

    mask = _random_block_mask(batch, cfg_ssl.mask_block_size, cfg_ssl.mask_ratio)
    v_masked = _apply_mask(batch, mask, mask_token)
    z = _encode_deepest(encoder, v_masked)
    recon = decoder(z)

    L_recon = reconstruction_loss(recon, batch, mask, cfg_ssl.loss_kind)
    return {
        "ssl_loss": float(L_recon.item()),
        "recon":    float(L_recon.item()),
    }


### 7.6 `update_ema`

In [ ]:
# ===  EMA target encoder ===
# only calls this if `ssl_cfg.use_ema` is True.

def update_ema(online: nn.Module, target: nn.Module, momentum: float) -> None:
    """In-place EMA update: target = momentum * target + (1 - momentum) * online."""
    with torch.no_grad():
        for p_t, p_o in zip(target.parameters(), online.parameters()):
            p_t.data.mul_(momentum).add_(p_o.data, alpha=1.0 - momentum)
        for b_t, b_o in zip(target.buffers(), online.buffers()):
            b_t.data.copy_(b_o.data)


## 8. Validity / Sanity Checks

Method-agnostic checks that catch the silent bugs


In [ ]:
# --- Check 1: dataloader yields (view1, view2) of correct shape/dtype ---
batch = next(iter(train_loader))
assert isinstance(batch, (list, tuple)) and len(batch) == 2, (
    f"Expected (view1, view2) pair, got {type(batch).__name__} with len {len(batch)}"
)
v1, v2 = batch
assert v1.shape == v2.shape, f"View shape mismatch: {v1.shape} vs {v2.shape}"
assert v1.shape[1:] == (model_cfg.in_channels, model_cfg.img_size, model_cfg.img_size), (
    f"Bad view shape: {v1.shape}"
)
assert v1.dtype == torch.float32, f"Expected float32, got {v1.dtype}"
assert torch.isfinite(v1).all() and torch.isfinite(v2).all(), "Non-finite values in views"
print(f"[ok] views shape={tuple(v1.shape)} dtype={v1.dtype}")


In [ ]:
# --- Check 2: the two views actually differ (augmentation is doing something) ---
diff = (v1 - v2).abs().mean().item()
assert diff > 1e-3, (
    f"view1 and view2 are identical (mean abs diff = {diff:.2e}). "
    "Augmentation is not randomizing per view."
)
print(f"[ok] view1 vs view2 mean abs diff = {diff:.4f}")


In [ ]:
# --- Check 3: channel-order invariant ---
expected = ["pre_synaptic", "post_synaptic", "structural"]
assert list(data_cfg.channel_names) == expected, (
    f"Channel order mismatch. data_cfg.channel_names = {data_cfg.channel_names}, "
    f"expected {expected}. Channels carry distinct biological meaning — never permute."
)
print(f"[ok] channel order: {data_cfg.channel_names}")


In [ ]:
# --- Check 4: encoder forward + per-stage feature shapes ---
encoder.eval()
with torch.no_grad():
    feats = encoder(v1[:2].to(device).contiguous())
assert isinstance(feats, list) and len(feats) == 5, (
    f"Expected 5 stage outputs from MONAI SwinTransformer, got {type(feats).__name__} with {len(feats) if hasattr(feats, '__len__') else '?'}"
)
print("[ok] encoder forward — per-stage feature shapes:")
for i, f in enumerate(feats):
    print(f"     stage {i}: {tuple(f.shape)}")
encoder.train()


In [ ]:
# --- Check 5: pretrained weights actually loaded (not silently random) ---
# A randomly-initialised conv kernel has near-zero mean and a fairly large std
# determined by Kaiming init. Pretrained ImageNet kernels deviate from that.
first_conv = None
for m in encoder.modules():
    if isinstance(m, nn.Conv2d):
        first_conv = m
        break
assert first_conv is not None, "No Conv2d found in encoder"
w = first_conv.weight.detach().cpu()
print(f"[info] first conv kernel: shape={tuple(w.shape)} "
      f"mean={w.mean():.4f} std={w.std():.4f} "
      f"min={w.min():.4f} max={w.max():.4f}")

if cfg.init_source != "scratch":
    # A properly loaded ImageNet/SimMIM kernel has structured filters; its
    # std-of-channel-means is non-trivially large compared to a fresh kaiming_normal.
    channel_mean_std = w.mean(dim=(2, 3)).std().item()
    print(f"[info] std of per-(out,in) kernel means = {channel_mean_std:.5f}")
    if load_summary["n_loaded"] == 0:
        raise RuntimeError(
            "init_source != 'scratch' but no parameters were loaded. "
            "Inspect load_summary."
        )
    if load_summary["n_loaded"] < 0.5 * load_summary["n_target_params"]:
        warnings.warn("Less than half of encoder params were loaded — see load_summary.")
print(f"[ok] init_source='{cfg.init_source}' verified")


In [ ]:
# --- Check 6: visualization (4 random samples) ---
fig, axes = plt.subplots(4, 2 * model_cfg.in_channels, figsize=(2 * 2 * model_cfg.in_channels, 8))
for row in range(4):
    raw = train_subset[row]  # (C,H,W) in [0,1]
    a, b = train_transform(raw)
    a = a.cpu().numpy()
    b = b.cpu().numpy()
    for ci, name in enumerate(data_cfg.channel_names):
        axes[row, ci].imshow(a[ci], cmap="magma")
        axes[row, ci].set_title(f"v1 {name}" if row == 0 else "")
        axes[row, ci].axis("off")
        axes[row, model_cfg.in_channels + ci].imshow(b[ci], cmap="magma")
        axes[row, model_cfg.in_channels + ci].set_title(f"v2 {name}" if row == 0 else "")
        axes[row, model_cfg.in_channels + ci].axis("off")
plt.suptitle("Two-view augmentation — 4 random training patches")
plt.tight_layout()
plt.show()


In [ ]:
# --- Check 7: post-normalization histograms per channel (sanity for ch_stats) ---
fig, axes = plt.subplots(1, model_cfg.in_channels, figsize=(4 * model_cfg.in_channels, 3))
for ci, name in enumerate(data_cfg.channel_names):
    pix = v1[:, ci].flatten().cpu().numpy()
    axes[ci].hist(pix, bins=80, color="steelblue", alpha=0.85)
    axes[ci].set_title(f"{name}\nmean={pix.mean():.2f}  std={pix.std():.2f}")
    axes[ci].axvline(0, color="k", linestyle="--", linewidth=0.8)
plt.suptitle("Post-normalization channel histograms (should be ~zero-mean, ~unit-std)")
plt.tight_layout()
plt.show()


In [ ]:
# --- Check 8: dummy fwd+bwd step---
encoder.train()
opt = torch.optim.AdamW(encoder.parameters(), lr=1e-4)
v1_d = v1.to(device).contiguous()
v2_d = v2.to(device).contiguous()
opt.zero_grad(set_to_none=True)
out = encoder(v1_d)[4].mean() + encoder(v2_d)[4].mean()
out.backward()
n_with_grad = sum(1 for p in encoder.parameters() if p.grad is not None)
n_total = sum(1 for _ in encoder.parameters())
opt.step()
del opt
print(f"[ok] dummy fwd+bwd succeeded — {n_with_grad}/{n_total} params received gradients")


## 9. Logging Setup

CSV logger and experiment metadata. Re-run after changing `cfg`/`train_cfg`
to start a fresh `save_dir`.


In [ ]:
run_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
save_dir = os.path.join(
    cfg.output_root,
    f"{train_cfg.experiment_name}_{cfg.method_name}_{run_timestamp}"
)
os.makedirs(save_dir, exist_ok=True)
print(f"Output directory: {save_dir}")

# Persist channel statistics next to the checkpoints.
stats = {
    "channel_names": list(data_cfg.channel_names),
    "mean": CH_MEAN.tolist(),
    "std": CH_STD.tolist(),
}
data_cfg.channel_stats_path = os.path.join(save_dir, "channel_stats.json")
with open(data_cfg.channel_stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"Channel stats saved to: {data_cfg.channel_stats_path}")


In [ ]:
experiment_meta = {
    "experiment_name":   train_cfg.experiment_name,
    "method_name":       cfg.method_name,
    "init_source":       cfg.init_source,
    "tag":               cfg.tag,
    "start_time":        datetime.now().isoformat(),
    "device":            str(device),
    "gpu_name":          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "pytorch_version":   torch.__version__,

    "n_train":           len(train_loader.dataset),
    "n_val":             len(val_loader.dataset),
    "n_train_batches":   len(train_loader),
    "n_val_batches":     len(val_loader),
    "img_size":          model_cfg.img_size,
    "in_channels":       model_cfg.in_channels,
    "channel_names":     list(data_cfg.channel_names),
    "channel_mean":      CH_MEAN.tolist(),
    "channel_std":       CH_STD.tolist(),

    "encoder":           "SwinTransformer (MONAI, spatial_dims=2)",
    "feature_size":      model_cfg.feature_size,
    "depths":            list(model_cfg.depths),
    "num_heads":         list(model_cfg.num_heads),
    "window_size":       model_cfg.window_size,
    "patch_size":        model_cfg.patch_size,
    "encoder_params":    sum(p.numel() for p in encoder.parameters()),
    "init_summary":      {
        k: v for k, v in load_summary.items()
        if k in ("source", "n_loaded", "n_random_init", "n_skipped_in_source", "n_shape_mismatch")
    },

    "optimizer":         "AdamW (LLRD on encoder)",
    "base_lr":           train_cfg.base_lr,
    "head_lr":           train_cfg.head_lr,
    "weight_decay":      train_cfg.weight_decay,
    "layer_decay":       train_cfg.layer_decay,
    "batch_size":        data_cfg.batch_size,
    "epochs":            train_cfg.epochs,
    "warmup_epochs":     train_cfg.warmup_epochs,
    "freeze_encoder_epochs": train_cfg.freeze_encoder_epochs,
    "scheduler":         "linear warmup + cosine decay",
    "mixed_precision":   device.type == "cuda",
    "grad_clip_norm":    train_cfg.grad_clip_norm,
    "seed":              cfg.seed,
}

print("=" * 60)
for k, v in experiment_meta.items():
    print(f"  {k:24s}: {v}")
print("=" * 60)

with open(os.path.join(save_dir, "experiment_meta.json"), "w") as f:
    json.dump(experiment_meta, f, indent=2, default=str)

csv_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_log.csv")
csv_fields = [
    "epoch", "phase", "train_loss", "val_metric", "lr_encoder", "lr_head",
    "epoch_time_s", "train_time_s", "val_time_s",
    "best_val_metric", "best_epoch",
    "grad_norm_mean", "grad_norm_max",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields, extrasaction="ignore")
csv_writer.writeheader()
print(f"Logging to: {csv_path}")


## 10. Checkpoint Save / Load Helpers


In [ ]:
def _heads_state_dict(heads):
    if heads is None:
        return None
    if isinstance(heads, nn.Module):
        return heads.state_dict()
    if isinstance(heads, dict):
        return {k: v.state_dict() for k, v in heads.items()}
    raise TypeError(f"Unsupported heads type: {type(heads)}")


def _load_heads_state_dict(heads, sd):
    if sd is None or heads is None:
        return
    if isinstance(heads, nn.Module):
        heads.load_state_dict(sd)
    else:
        for k, mod in heads.items():
            mod.load_state_dict(sd[k])


def save_checkpoint(path, *, encoder, heads, optimizer, scheduler, scaler,
                    epoch, val_metric, train_loss, extra: dict | None = None):
    payload = {
        "epoch":                epoch,
        "encoder_state_dict":   encoder.state_dict(),
        "heads_state_dict":     _heads_state_dict(heads),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict":    scaler.state_dict() if scaler is not None else None,
        "val_metric":           val_metric,
        "train_loss":           train_loss,
        "model_cfg":            vars(model_cfg),
        "train_cfg":            {k: v for k, v in vars(train_cfg).items()
                                 if isinstance(v, (int, float, str, bool, tuple, list))},
        "ssl_cfg":              {k: v for k, v in vars(ssl_cfg).items()
                                 if isinstance(v, (int, float, str, bool, tuple, list))},
        "init_source":          cfg.init_source,
        "method_name":          cfg.method_name,
        "channel_mean":         CH_MEAN.tolist(),
        "channel_std":          CH_STD.tolist(),
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_checkpoint(path, *, encoder, heads, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state_dict"])
    _load_heads_state_dict(heads, ckpt.get("heads_state_dict"))
    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt


In [ ]:
# Optional resume. Leave `resume_path = None` for fresh training.
resume_path = None
start_epoch = 1

# Best-metric tracker, set after we know the direction.
_better = (lambda new, best: new < best) if train_cfg.val_metric_direction == "min"           else (lambda new, best: new > best)
_init_best = float("inf") if train_cfg.val_metric_direction == "min" else float("-inf")


## 11. Smoke Run

Trains for `train_cfg.smoke_epochs` epoch(s) on `train_cfg.smoke_n'


In [ ]:
def smoke_run():
    print(f"[smoke] {train_cfg.smoke_epochs} epoch(s) × {train_cfg.smoke_n} samples")
    smoke_subset = Subset(train_subset, list(range(min(train_cfg.smoke_n, len(train_subset)))))
    smoke_ds = _AugmentedSubset(smoke_subset, train_transform)
    smoke_loader = DataLoader(
        smoke_ds, batch_size=min(8, len(smoke_ds)), shuffle=True,
        num_workers=0, drop_last=True,
    )

    enc = build_swin_encoder(model_cfg).to(device)
    load_pretrained_into_encoder(enc, cfg.init_source, model_cfg)

    try:
        heads = build_heads(enc, ssl_cfg)
    except NotImplementedError as e:
        print(f"[smoke] build_heads not implemented: {e}")
        return False
    if isinstance(heads, nn.Module):
        heads = heads.to(device)
    elif isinstance(heads, dict):
        heads = {k: v.to(device) for k, v in heads.items()}

    try:
        opt, sched = build_optimizer_and_scheduler(enc, heads, train_cfg)
    except NotImplementedError as e:
        print(f"[smoke] build_optimizer_and_scheduler not implemented: {e}")
        return False

    enc.train()
    if isinstance(heads, nn.Module): heads.train()
    elif isinstance(heads, dict):    [m.train() for m in heads.values()]

    for ep in range(train_cfg.smoke_epochs):
        for v1_, v2_ in smoke_loader:
            v1_ = v1_.to(device); v2_ = v2_.to(device)
            opt.zero_grad(set_to_none=True)
            try:
                loss, metrics = compute_loss(enc, heads, v1_, v2_, ssl_cfg)
            except NotImplementedError as e:
                print(f"[smoke] compute_loss not implemented: {e}")
                return False
            loss.backward()
            opt.step()
        sched.step()
        print(f"[smoke] epoch {ep+1}: loss={loss.item():.4f}  metrics={metrics}")

    # validation_step
    enc.eval()
    if isinstance(heads, nn.Module): heads.eval()
    elif isinstance(heads, dict):    [m.eval() for m in heads.values()]
    with torch.no_grad():
        val_batch = val_transform(val_subset[0]).unsqueeze(0).to(device)
        try:
            vmetrics = validation_step(enc, heads, val_batch, ssl_cfg)
        except NotImplementedError as e:
            print(f"[smoke] validation_step not implemented: {e}")
            return False
    print(f"[smoke] validation_step OK  metrics={vmetrics}")
    assert train_cfg.val_metric_key in vmetrics, (
        f"validation_step must return train_cfg.val_metric_key='{train_cfg.val_metric_key}'; "
        f"got keys={list(vmetrics)}"
    )
    print("[smoke] OK")
    return True


# smoke_run()


## 11.5 Overfit-on-batch sanity check + reconstruction visualisation

End-to-end sanity check before launching the real training run. Trains a
**temporary** copy of (encoder, decoder, mask_token, projector) on the
same 4 train patches for ~`N_OVERFIT_STEPS` steps and visualises what the
decoder reconstructs at masked positions. If the reconstruction loss does
not collapse on 4 patches in this budget, the model + loss is broken
regardless of how the full training looks.

The block snapshots `encoder.state_dict()` (CPU clones, no GPU spike)
and the global RNG state on entry, and restores both before exiting so
that the main training run (Section 12) starts from byte-identical state
to a notebook that skipped this section. The temporary heads are dropped.

The overfit loop uses a **fixed** augmented batch (v1, v2) and a
**fixed** mask, so the recon and VICReg terms are deterministic functions
of the network parameters — a real overfit, not training on a moving
target. Note: this checks only the *reconstruction path* and that VICReg
does not collapse to a trivial constant;

In [ ]:
# === SIMMIM-VICREG-OVERFIT-ON-BATCH ===
import random as _random_mod

RUN_OVERFIT_CHECK = True   # set to False to skip the entire 11.5 block
N_OVERFIT_STEPS   = 500    # decoder + encoder converge fast on 4 patches
OVERFIT_LR        = 1e-4

overfit_cfg = SimpleNamespace(
    patch_indices=[3, 4, 50, min(1000, len(train_subset) - 1)],
    n_unseen=3,
    seed=cfg.seed + 17,
)


def _move_to_device(d, dev):
    """Local mini-helper; the global `_move_heads_to` is defined later
    in the training-loop section."""
    if isinstance(d, nn.Module): return d.to(dev)
    if isinstance(d, dict):      return {k: v.to(dev) for k, v in d.items()}
    return d


def _fixed_batch_from_subset(subset, indices, transform, seed):
    """Apply `transform` once per index under a saved/restored RNG state.

    Returns: (view1_stack, view2_stack) — fixed reproducibly across re-runs.
    The outer RNG is unchanged, so calling this is safe wrt the main run.
    """
    cpu_state  = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    np_state   = np.random.get_state()
    py_state   = _random_mod.getstate()
    try:
        torch.manual_seed(seed); np.random.seed(seed); _random_mod.seed(seed)
        v1, v2 = [], []
        for idx in indices:
            a, b = transform(subset[idx])
            v1.append(a); v2.append(b)
        return torch.stack(v1), torch.stack(v2)
    finally:
        torch.set_rng_state(cpu_state)
        if cuda_state is not None: torch.cuda.set_rng_state_all(cuda_state)
        np.random.set_state(np_state)
        _random_mod.setstate(py_state)


# --- Snapshot the global RNG and encoder state so the *real* training run
# is byte-identical to what it would be without this overfit cell.
_rng_cpu_state  = torch.get_rng_state()
_rng_cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
_rng_np_state   = np.random.get_state()
_rng_py_state   = _random_mod.getstate()
# CPU clones rather than deepcopy of GPU tensors (avoids OOM spikes).
_saved_encoder  = {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()}

# Cache fixed (view1, view2) and a fixed mask so the overfit target is
# truly stationary. compute_loss internally re-samples the mask per step,
# so we inline the loss math below using `mask_fixed` (mirrors compute_loss
# exactly otherwise).
x1_fixed, x2_fixed = _fixed_batch_from_subset(
    train_subset, overfit_cfg.patch_indices, train_transform, overfit_cfg.seed,
)
x1_fixed = x1_fixed.to(device); x2_fixed = x2_fixed.to(device)
B_fixed = x1_fixed.shape[0]
torch.manual_seed(overfit_cfg.seed + 1)  # deterministic mask draw
mask_fixed = _random_block_mask(
    x1_fixed, ssl_cfg.mask_block_size, ssl_cfg.mask_ratio,
).clone()
print(f"fixed overfit batch: {tuple(x1_fixed.shape)}  "
      f"indices={overfit_cfg.patch_indices}  "
      f"mask_fraction={mask_fixed.mean().item():.3f}")


if RUN_OVERFIT_CHECK:
    overfit_heads = _move_to_device(build_heads(encoder, ssl_cfg), device)

    head_params = (
        list(overfit_heads.parameters())
        if isinstance(overfit_heads, nn.Module)
        else [p for h in overfit_heads.values() for p in h.parameters()]
    )
    opt = torch.optim.AdamW(
        list(encoder.parameters()) + head_params,
        lr=OVERFIT_LR, weight_decay=0.0,
    )

    encoder.train()
    if isinstance(overfit_heads, nn.Module): overfit_heads.train()
    else:                                    [m.train() for m in overfit_heads.values()]

    decoder    = overfit_heads["decoder"]
    mask_token = overfit_heads["mask_token"]
    projector  = overfit_heads["projector"]
    rec_w  = float(getattr(ssl_cfg, "w_recon", 1.0))
    vic_w  = float(getattr(ssl_cfg, "w_vicreg", 1.0))

    history = {"loss": [], "recon": [], "sim": [], "std": [], "cov": []}
    for step in range(N_OVERFIT_STEPS):
        # Inline loss with FIXED mask (deterministic given parameters).
        v1m = _apply_mask(x1_fixed, mask_fixed, mask_token)
        v2m = _apply_mask(x2_fixed, mask_fixed, mask_token)
        z1  = encoder(v1m.contiguous())[-1]
        z2  = encoder(v2m.contiguous())[-1]
        r1  = decoder(z1); r2 = decoder(z2)
        L_recon = 0.5 * (
            reconstruction_loss(r1, x1_fixed, mask_fixed, ssl_cfg.loss_kind)
            + reconstruction_loss(r2, x2_fixed, mask_fixed, ssl_cfg.loss_kind)
        )
        p1 = projector(z1.mean(dim=(-2, -1)).float())
        p2 = projector(z2.mean(dim=(-2, -1)).float())
        L_sim, L_std, L_cov = _vicreg_terms(p1, p2)
        L_vicreg = (ssl_cfg.lambda_sim * L_sim
                    + ssl_cfg.lambda_std * L_std
                    + ssl_cfg.lambda_cov * L_cov)
        loss = rec_w * L_recon + vic_w * L_vicreg

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + head_params, max_norm=5.0,
        )
        opt.step()

        history["loss"].append(float(loss.item()))
        history["recon"].append(float(L_recon.item()))
        history["sim"].append(float(L_sim.item()))
        history["std"].append(float(L_std.item()))
        history["cov"].append(float(L_cov.item()))

    fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
    eps = 1e-12
    for k, vs in history.items():
        if any(v > eps for v in vs):
            ax.plot([max(v, eps) for v in vs], label=k)
    ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("loss / term (log)")
    ax.legend(loc="upper right")
    ax.set_title(
        f"SimMIM+VICReg overfit-on-{B_fixed}-patches  "
        f"recon: {history['recon'][0]:.4f} -> {history['recon'][-1]:.4f}"
    )
    plt.tight_layout(); plt.show()

    drop = (history["recon"][0] - history["recon"][-1]) / max(history["recon"][0], 1e-8)
    print(f"recon drop over {N_OVERFIT_STEPS} steps: {drop*100:.1f}%")
    if drop < 0.50:
        print("  WARNING: model failed to overfit a tiny batch; investigate "
              "loss / decoder / fp16 overflow / encoder freeze before "
              "launching full training.")
    print("  Note: this checks the *reconstruction path*; it does NOT prove "
          "the encoder learned useful features for downstream fine-tuning.")
else:
    overfit_heads = None
    print("overfit-on-batch sanity check skipped (RUN_OVERFIT_CHECK=False)")


In [ ]:
# === SIMMIM-VICREG-OVERFIT-RECON-VIZ ===

_CH_MEAN_DEV = CH_MEAN.to(device).view(1, -1, 1, 1)
_CH_STD_DEV  = CH_STD.to(device).view(1, -1, 1, 1)


def _denorm(x):
    """Undo the per-channel z-score so plots are back in image space."""
    return x * _CH_STD_DEV + _CH_MEAN_DEV


@torch.no_grad()
def _sv_recon(encoder, heads, view, mask):
    """Forward (encoder + heads['decoder']) on the masked view. eval()."""
    encoder.eval()
    if isinstance(heads, nn.Module): heads.eval()
    else:                            [m.eval() for m in heads.values()]
    v_masked = _apply_mask(view, mask, heads["mask_token"])
    z = encoder(v_masked.contiguous())[-1]
    recon = heads["decoder"](z)
    return recon.float(), v_masked.float()


def _viz_sv_recon(view, recon, masked_view, mask, indices, suptitle):
    """4 rows x N cols: input | masked input | recon | |err| (channel-mean).

    `view`, `recon`, `masked_view` are denormalised for display
    (clamped to [0, 1] only for imshow). The error map is computed from
    the unclamped denormalised tensors so a recon that overshoots [0, 1]
    still surfaces as bright error. `mask` is overlaid as a faint cyan
    outline on the masked-input row to emphasise which pixels were hidden.
    """
    n = view.shape[0]
    inp_im   = _denorm(view).cpu()
    rec_im   = _denorm(recon).cpu()
    mask_im  = _denorm(masked_view).cpu()
    mask_arr = mask.cpu().squeeze(1).numpy()           # (N, H, W)
    err_im   = (rec_im - inp_im).abs().mean(dim=1)     # (N, H, W)
    inp_disp  = inp_im.clamp(0, 1)
    rec_disp  = rec_im.clamp(0, 1)
    mask_disp = mask_im.clamp(0, 1)

    def _to_disp(t):
        return t.permute(1, 2, 0).numpy() if t.shape[0] == 3 else t.mean(0).numpy()

    fig, axes = plt.subplots(4, n, figsize=(3.2 * n, 12.0), squeeze=False)
    for j in range(n):
        axes[0, j].imshow(_to_disp(inp_disp[j]))
        axes[0, j].set_title(f"input idx={indices[j]}"); axes[0, j].axis("off")
        axes[1, j].imshow(_to_disp(mask_disp[j]))
        axes[1, j].contour(mask_arr[j], levels=[0.5], colors=["c"], linewidths=0.7)
        axes[1, j].set_title(f"masked  ({mask_arr[j].mean()*100:.0f}% hidden)")
        axes[1, j].axis("off")
        axes[2, j].imshow(_to_disp(rec_disp[j]))
        axes[2, j].set_title("recon"); axes[2, j].axis("off")
        axes[3, j].imshow(err_im[j].numpy(), cmap="hot")
        axes[3, j].set_title(f"|err|  mean={err_im[j].mean().item():.3f}")
        axes[3, j].axis("off")
    plt.suptitle(suptitle); plt.tight_layout(); plt.show()


if RUN_OVERFIT_CHECK and overfit_heads is not None:
    # --- pick `n_unseen` unseen random train patches (deterministic).
    _excluded = set(int(i) for i in overfit_cfg.patch_indices)
    _rng = np.random.default_rng(overfit_cfg.seed + 23)
    _pool = [i for i in range(len(train_subset)) if i not in _excluded]
    other_indices = [int(x) for x in _rng.choice(
        _pool, size=min(overfit_cfg.n_unseen, len(_pool)), replace=False,
    )]
    print(f"unseen train patches: {other_indices}  (out of {len(train_subset)})")

    x1_other, _ = _fixed_batch_from_subset(
        train_subset, other_indices, train_transform, overfit_cfg.seed + 1,
    )
    x1_other = x1_other.to(device)
    # Independent mask for the unseen-patch batch (same seed offset for repro).
    torch.manual_seed(overfit_cfg.seed + 31)
    mask_other = _random_block_mask(
        x1_other, ssl_cfg.mask_block_size, ssl_cfg.mask_ratio,
    ).clone()

    # --- inference; encoder + overfit_heads still in their overfitted state.
    recon_fixed, v1_masked_fixed = _sv_recon(encoder, overfit_heads, x1_fixed, mask_fixed)
    recon_other, v1_masked_other = _sv_recon(encoder, overfit_heads, x1_other, mask_other)

    print("\nOVERFIT patches (expect near-perfect masked-pixel recon):")
    for j, idx in enumerate(overfit_cfg.patch_indices):
        per_pix = (recon_fixed[j:j+1] - x1_fixed[j:j+1]).abs() * mask_fixed[j:j+1]
        denom = mask_fixed[j:j+1].sum() * x1_fixed.shape[1] + 1e-8
        l1 = float(per_pix.sum() / denom)
        print(f"  overfit_idx={idx}: masked-L1={l1:.5f}")
    _viz_sv_recon(
        x1_fixed, recon_fixed, v1_masked_fixed, mask_fixed,
        overfit_cfg.patch_indices,
        f"Overfit-trained recon on the {B_fixed} TRAIN patches",
    )

    print("\nUNSEEN train patches (expect poor recon -- 4-patch overfit, OOD):")
    for j, idx in enumerate(other_indices):
        per_pix = (recon_other[j:j+1] - x1_other[j:j+1]).abs() * mask_other[j:j+1]
        denom = mask_other[j:j+1].sum() * x1_other.shape[1] + 1e-8
        l1 = float(per_pix.sum() / denom)
        print(f"  unseen_idx={idx}: masked-L1={l1:.5f}")
    _viz_sv_recon(
        x1_other, recon_other, v1_masked_other, mask_other,
        other_indices,
        f"Overfit-trained recon on {x1_other.shape[0]} UNSEEN train patches",
    )
else:
    print("overfit-recon visualisation skipped (RUN_OVERFIT_CHECK=False)")


# --- Restore: encoder weights + RNG state. The overfit_heads are dropped
# so the next cell builds fresh `heads`. After this block, the training
# run starts byte-identical to a notebook that skipped Section 11.5.
encoder.load_state_dict(_saved_encoder)
torch.set_rng_state(_rng_cpu_state)
if _rng_cuda_state is not None: torch.cuda.set_rng_state_all(_rng_cuda_state)
np.random.set_state(_rng_np_state)
_random_mod.setstate(_rng_py_state)
del _saved_encoder
if overfit_heads is not None:
    del overfit_heads
print("[ok] encoder + RNG state restored to pre-overfit values")


## 12. Training Loop

Two-phase: encoder is frozen for `train_cfg.freeze_encoder_epochs` epochs so
the heads can warm up against pretrained features without destroying them,
then the encoder is unfrozen for the remainder. AMP autocast on CUDA,
gradient clipping, layer-wise LR decay,
periodic best-checkpoint save.


In [ ]:
def freeze_encoder(enc, freeze: bool):
    for p in enc.parameters():
        p.requires_grad = not freeze


def _move_heads_to(heads, dev):
    if isinstance(heads, nn.Module):
        return heads.to(dev)
    if isinstance(heads, dict):
        return {k: v.to(dev) for k, v in heads.items()}
    return heads


def _set_train(enc, heads, training: bool):
    enc.train(training)
    if isinstance(heads, nn.Module):
        heads.train(training)
    elif isinstance(heads, dict):
        for m in heads.values(): m.train(training)


def _heads_iter_lrs(optimizer):
    """Return (encoder_lr, head_lr) heuristically for logging."""
    lrs = [g["lr"] for g in optimizer.param_groups]
    if not lrs:
        return 0.0, 0.0
    # Encoder groups have key "stage" (set by param_groups_layer_decay);
    # the head group does not.
    enc_lrs = [g["lr"] for g in optimizer.param_groups if "stage" in g]
    head_lrs = [g["lr"] for g in optimizer.param_groups if "stage" not in g]
    enc_lr = max(enc_lrs) if enc_lrs else 0.0
    head_lr = head_lrs[0] if head_lrs else 0.0
    return enc_lr, head_lr


In [ ]:
# Build heads + optimizer 
heads = build_heads(encoder, ssl_cfg)
heads = _move_heads_to(heads, device)
optimizer, scheduler = build_optimizer_and_scheduler(encoder, heads, train_cfg)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

# Optional EMA target encoder. Only created if user wants it.
ema_encoder = None
if getattr(ssl_cfg, "use_ema", False):
    ema_encoder = build_swin_encoder(model_cfg).to(device)
    ema_encoder.load_state_dict(encoder.state_dict())
    for p in ema_encoder.parameters():
        p.requires_grad = False

if resume_path is not None:
    ckpt = load_checkpoint(resume_path,
                           encoder=encoder, heads=heads,
                           optimizer=optimizer, scheduler=scheduler, scaler=scaler)
    start_epoch = ckpt["epoch"] + 1
    best_val_metric = ckpt["val_metric"]
    best_epoch = ckpt["epoch"]
    print(f"Resumed from epoch {ckpt['epoch']}, val_metric={ckpt['val_metric']:.6f}")
else:
    best_val_metric = _init_best
    best_epoch = 0
    start_epoch = 1


In [ ]:
train_losses, val_metrics_hist, lr_history, grad_norms, epoch_times = [], [], [], [], []
total_train_start = time.time()

if cfg.dry_run:
    print("dry_run=True — skipping full training loop.")
else:
    for epoch in range(start_epoch, train_cfg.epochs + 1):
        # --- Phase control ---
        in_warmup_phase = epoch <= train_cfg.freeze_encoder_epochs
        freeze_encoder(encoder, freeze=in_warmup_phase)
        phase = "frozen" if in_warmup_phase else "full"

        # --- Train ---
        _set_train(encoder, heads, True)
        running_loss = 0.0
        epoch_grad_norms = []
        train_start = time.time()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{train_cfg.epochs} [{phase}]", leave=False)
        for v1_b, v2_b in pbar:
            v1_b = v1_b.to(device, non_blocking=True)
            v2_b = v2_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                loss, metrics = compute_loss(encoder, heads, v1_b, v2_b, ssl_cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            total_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in encoder.parameters() if p.requires_grad]
                + ([p for p in (heads.parameters() if isinstance(heads, nn.Module)
                                else (q for h in heads.values() for q in h.parameters()))]),
                max_norm=train_cfg.grad_clip_norm,
            )
            epoch_grad_norms.append(total_norm.item())
            scaler.step(optimizer)
            scaler.update()
            if ema_encoder is not None:
                update_ema(encoder, ema_encoder, getattr(ssl_cfg, "ema_momentum", 0.996))
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss = running_loss / max(1, len(train_loader))
        train_time = time.time() - train_start
        scheduler.step()

        # --- Validation ---
        _set_train(encoder, heads, False)
        val_metric_accum = {}
        val_start = time.time()
        with torch.no_grad():
            n_val_batches = 0
            for v_b in val_loader:
                if isinstance(v_b, (list, tuple)):
                    v_b = v_b[0]
                v_b = v_b.to(device, non_blocking=True)
                vmetrics = validation_step(encoder, heads, v_b, ssl_cfg)
                for k, v in vmetrics.items():
                    val_metric_accum[k] = val_metric_accum.get(k, 0.0) + float(v)
                n_val_batches += 1
            for k in val_metric_accum:
                val_metric_accum[k] /= max(1, n_val_batches)
        val_time = time.time() - val_start
        val_metric = val_metric_accum[train_cfg.val_metric_key]

        # --- Logging ---
        enc_lr, head_lr = _heads_iter_lrs(optimizer)
        lr_history.append((enc_lr, head_lr))
        epoch_time = train_time + val_time
        epoch_times.append(epoch_time)
        mean_grad = sum(epoch_grad_norms) / max(1, len(epoch_grad_norms))
        max_grad  = max(epoch_grad_norms) if epoch_grad_norms else 0.0
        grad_norms.append(mean_grad)
        train_losses.append(train_loss)
        val_metrics_hist.append(val_metric_accum)

        improved = ""
        if _better(val_metric, best_val_metric):
            best_val_metric = val_metric
            best_epoch = epoch
            improved = " *best*"
            save_checkpoint(
                os.path.join(save_dir, "best_model.pt"),
                encoder=encoder, heads=heads, optimizer=optimizer,
                scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
                extra={"all_val_metrics": val_metric_accum},
            )

        print(f"Epoch {epoch:3d}/{train_cfg.epochs} [{phase}] "
              f"| Train: {train_loss:.6f} "
              f"| Val[{train_cfg.val_metric_key}]: {val_metric:.6f} "
              f"| LR(enc/head): {enc_lr:.2e}/{head_lr:.2e} "
              f"| Time: {epoch_time:.1f}s "
              f"| GradNorm: {mean_grad:.3f}{improved}")

        csv_writer.writerow({
            "epoch": epoch,
            "phase": phase,
            "train_loss": train_loss,
            "val_metric": val_metric,
            "lr_encoder": enc_lr,
            "lr_head": head_lr,
            "epoch_time_s": epoch_time,
            "train_time_s": train_time,
            "val_time_s": val_time,
            "best_val_metric": best_val_metric,
            "best_epoch": best_epoch,
            "grad_norm_mean": mean_grad,
            "grad_norm_max": max_grad,
        })
        csv_file.flush()

    total_train_time = time.time() - total_train_start
    csv_file.close()
    print(f"\nTotal training time: {total_train_time / 60:.1f} min")

## 12.5 Post-training reconstruction visualisation

Reload the best checkpoint and inspect what the SimMIM + VICReg decoder
learned. Renders three diagnostic groups:

  1. The same overfit indices used in Section 11.5 (same deterministic
     `train_transform` seed, same fixed mask) — directly comparable to
     the overfit-only panel above so you can see how full training
     generalises.
  2. A small batch of validation patches (single view, no augmentation,
     z-score only) with a fresh random mask — the cleanest measurement
     of in-distribution masked reconstruction.
  3. Loss curves from `{train_cfg.experiment_name}_log.csv`.


In [ ]:
# === SIMMIM-VICREG-POST-RECON-VIZ ===
import csv as _csv

_best_path = os.path.join(save_dir, "best_model.pt")
if not os.path.exists(_best_path):
    raise FileNotFoundError(f"no best checkpoint found at {_best_path}")

ckpt = torch.load(_best_path, map_location=device, weights_only=False)
encoder.load_state_dict(ckpt["encoder_state_dict"])
_load_heads_state_dict(heads, ckpt.get("heads_state_dict"))
print(f"reloaded {os.path.basename(_best_path)}: "
      f"epoch={ckpt.get('epoch', '?')}  "
      f"val[{train_cfg.val_metric_key}]="
      f"{float(ckpt.get('val_metric', float('nan'))):.5f}")

# (1) Overfit indices on the train split (same deterministic seed as 11.5)
_overfit_indices = (overfit_cfg.patch_indices if "overfit_cfg" in dir()
                    else [3, 4, 50, min(1000, len(train_subset) - 1)])
_overfit_seed    = overfit_cfg.seed if "overfit_cfg" in dir() else cfg.seed + 17

x_train_post, _ = _fixed_batch_from_subset(
    train_subset, _overfit_indices, train_transform, _overfit_seed,
)
x_train_post = x_train_post.to(device)
torch.manual_seed(_overfit_seed + 1)
_mask_train_post = _random_block_mask(
    x_train_post, ssl_cfg.mask_block_size, ssl_cfg.mask_ratio,
).clone()
recon_train_post, masked_train_post = _sv_recon(
    encoder, heads, x_train_post, _mask_train_post,
)
print("\nPOST: overfit indices on train split (matches Section 11.5 seed):")
for j, idx in enumerate(_overfit_indices):
    per_pix = (recon_train_post[j:j+1] - x_train_post[j:j+1]).abs() * _mask_train_post[j:j+1]
    denom = _mask_train_post[j:j+1].sum() * x_train_post.shape[1] + 1e-8
    l1 = float(per_pix.sum() / denom)
    print(f"  idx={idx}: masked-L1={l1:.5f}")
_viz_sv_recon(
    x_train_post, recon_train_post, masked_train_post, _mask_train_post,
    _overfit_indices,
    "POST: overfit indices on train split",
)

# (2) Validation patches (single view, no augmentation; cleanest signal).
_n_val_show  = min(4, len(val_subset))
_val_indices = list(range(_n_val_show))
x_val = torch.stack([val_transform(val_subset[i]) for i in _val_indices]).to(device)
torch.manual_seed(_overfit_seed + 2)
_mask_val = _random_block_mask(
    x_val, ssl_cfg.mask_block_size, ssl_cfg.mask_ratio,
).clone()
recon_val, masked_val = _sv_recon(encoder, heads, x_val, _mask_val)
print(f"\nPOST: first {_n_val_show} val patches (no augmentation):")
for j, idx in enumerate(_val_indices):
    per_pix = (recon_val[j:j+1] - x_val[j:j+1]).abs() * _mask_val[j:j+1]
    denom = _mask_val[j:j+1].sum() * x_val.shape[1] + 1e-8
    l1 = float(per_pix.sum() / denom)
    print(f"  val_idx={idx}: masked-L1={l1:.5f}")
_viz_sv_recon(
    x_val, recon_val, masked_val, _mask_val,
    _val_indices,
    f"POST: first {_n_val_show} validation patches (no aug)",
)

# (3) Loss / val_metric curves from CSV.
_csv_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_log.csv")
if os.path.exists(_csv_path):
    with open(_csv_path) as f:
        rows = list(_csv.DictReader(f))
    if rows:
        ep = [int(r["epoch"]) for r in rows]
        tr = [float(r["train_loss"]) for r in rows]
        vl = [float(r["val_metric"])  for r in rows]
        fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
        ax.plot(ep, tr, label="train_loss")
        ax.plot(ep, vl, label=f"val[{train_cfg.val_metric_key}]")
        ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
        ax.set_title("SimMIM+VICReg training curves"); ax.set_yscale("log")
        plt.tight_layout(); plt.show()
else:
    print(f"history CSV not found at {_csv_path}")


## 13. Post-Training: Embedding Inspection

Pulls pooled encoder embeddings on the validation set, plots a 2D PCA
(UMAP if installed), and reports effective rank + average pairwise cosine
similarity. These are universal "did the encoder learn anything?" checks
that work for any SSL method.


In [ ]:
@torch.no_grad()
def extract_pooled_embeddings(enc, loader, max_batches: int = 32) -> torch.Tensor:
    enc.eval()
    feats = []
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        if isinstance(batch, (list, tuple)):
            batch = batch[0]
        batch = batch.to(device, non_blocking=True).contiguous()
        z = enc(batch)[4]            # deepest stage
        z = z.mean(dim=(2, 3))       # global average pool
        feats.append(z.cpu())
    enc.train()
    return torch.cat(feats, dim=0)


# Load best checkpoint if it exists; otherwise inspect current weights.
best_path = os.path.join(save_dir, "best_model.pt")
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state_dict"])
    print(f"[inspect] loaded best_model.pt (epoch {ckpt['epoch']}, val_metric={ckpt['val_metric']:.6f})")

Z = extract_pooled_embeddings(encoder, val_loader, max_batches=32)
D = Z.shape[1]
print(f"[inspect] embeddings: N={Z.shape[0]}  D={D}")

# Effective rank (entropy of normalized squared singular values)
Zc = Z - Z.mean(dim=0, keepdim=True)
S = torch.linalg.svdvals(Zc.float())
s2 = (S ** 2) / (S ** 2).sum().clamp_min(1e-12)
eff_rank = torch.exp(-(s2 * torch.log(s2 + 1e-12)).sum()).item()
r_max = min(Z.shape[0] - 1, D)
print(f"[inspect] effective rank: {eff_rank:.1f} / {r_max} ({100 * eff_rank / r_max:.1f}%)")

# Mean pairwise cosine similarity (collapse indicator: ~1 == collapsed)
Zn = F.normalize(Z, dim=1)
n = min(1024, Z.shape[0])
sub = Zn[torch.randperm(Z.shape[0])[:n]]
cos = (sub @ sub.t())
mask = ~torch.eye(n, dtype=torch.bool)
mean_cos = cos[mask].mean().item()
print(f"[inspect] mean pairwise cosine similarity: {mean_cos:.4f}  "
      f"({'COLLAPSED' if mean_cos > 0.95 else 'OK'})")


In [ ]:
# 2D scatter — UMAP if available, else PCA.
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=cfg.seed)
    Z2 = reducer.fit_transform(Z.numpy())
    title = "UMAP of pooled encoder features"
except Exception:
    U, _, _ = torch.pca_lowrank(Z - Z.mean(0, keepdim=True), q=2)
    Z2 = (Z @ U).numpy()
    title = "PCA-2 of pooled encoder features"

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(Z2[:, 0], Z2[:, 1], s=5, alpha=0.6)
axes[0].set_title(title)
axes[0].set_xlabel("dim 1"); axes[0].set_ylabel("dim 2")
axes[1].plot((S / S.max()).numpy(), linewidth=1.5)
axes[1].set_yscale("log")
axes[1].set_title("Singular value spectrum (normalized)")
axes[1].set_xlabel("rank")
plt.tight_layout()
plt.show()


## 14. Save Encoder Weights

Strip optimizer / scheduler / scaler state and save just the encoder
state-dict so downstream fine-tuning notebooks can load it cleanly.


In [ ]:
best_path = os.path.join(save_dir, "best_model.pt")
if os.path.exists(best_path):
    best_ckpt = torch.load(best_path, map_location=device)
    encoder_path = os.path.join(save_dir, train_cfg.encoder_save_name)
    torch.save({
        "encoder_state_dict": best_ckpt["encoder_state_dict"],
        "model_cfg":          best_ckpt["model_cfg"],
        "channel_mean":       best_ckpt["channel_mean"],
        "channel_std":        best_ckpt["channel_std"],
        "init_source":        best_ckpt["init_source"],
        "method_name":        best_ckpt["method_name"],
        "epoch":              best_ckpt["epoch"],
        "val_metric":         best_ckpt["val_metric"],
    }, encoder_path)
    print(f"Saved encoder-only checkpoint: {encoder_path}")
else:
    print("No best_model.pt found — nothing to save.")
